In [ ]:
import numpy as np
import pandas as pd
from datetime import datetime, timedelta
import random

In [ ]:
# Set seed for reproducibility
np.random.seed(42)
random.seed(42)

N_VISITS = 10000

In [ ]:
# --- 1. Static/Tabular Data Generation ---

# Identifiers and Demographics
subject_ids = np.arange(10000, 10000 + N_VISITS)
stay_ids = np.arange(20000, 20000 + N_VISITS)
gender = np.random.choice(['M', 'F'], size=N_VISITS, p=[0.55, 0.45])
race = np.random.choice(['White', 'Black/African-American', 'Hispanic', 'Other/Unknown'],
                       size=N_VISITS, p=[0.5, 0.2, 0.15, 0.15])
age = np.random.randint(18, 91, size=N_VISITS)

# Timestamps
start_date = datetime(2150, 1, 1, 12, 0, 0)
intime = [start_date + timedelta(hours=np.random.randint(0, 500), minutes=np.random.randint(0, 60))
          for _ in range(N_VISITS)]
# ED stay duration: avg 5 hours + random variation
duration_hours = np.random.normal(5, 2, N_VISITS)
outtime = [intime[i] + timedelta(hours=duration_hours[i]) for i in range(N_VISITS)]

In [ ]:
# --- 2. Triage Target Generation (MTS Levels - Highly Imbalanced) ---

# Simulate MTS distribution: Critical (1/2) is rare, Urgent (3/4) is common.
# 1: <1%, 2: 5%, 3: 35%, 4: 45%, 5: 14%
acuity_distribution = [1, 2, 3, 4, 5]
acuity_p = [0.005, 0.05, 0.35, 0.45, 0.145]
acuity = np.random.choice(acuity_distribution, size=N_VISITS, p=acuity_p)

# ICU Admission (Rare Outcome)
# Flag is set to 1 only for a small fraction of Level 1/2/3 patients
icu_stay_flag = np.zeros(N_VISITS, dtype=int)
critical_indices = np.where((acuity <= 3) & (np.random.rand(N_VISITS) < 0.08))[0] # 8% chance for critical patients
icu_stay_flag[critical_indices] = 1
time_to_icu_hrs = np.where(icu_stay_flag == 1,
                           np.round(np.random.uniform(0.5, 10), 1), # ICU transfer within 0.5-10 hrs
                           None)


In [ ]:
# --- 3. Clinical & Time-Series Data Generation ---

# Establish baseline health based on acuity
def generate_vitals(acuity_level):
    if acuity_level == 1: # RED - Critical
        return {
            'hr_initial': int(np.random.normal(125, 15)), 'rr_initial': int(np.random.normal(28, 5)),
            'sbp_min': int(np.random.normal(70, 5)), 'o2sat_min': int(np.random.normal(88, 3))
        }
    elif acuity_level == 2: # ORANGE - Very Urgent
        return {
            'hr_initial': int(np.random.normal(110, 10)), 'rr_initial': int(np.random.normal(22, 4)),
            'sbp_min': int(np.random.normal(95, 10)), 'o2sat_min': int(np.random.normal(92, 2))
        }
    else: # YELLOW, GREEN, BLUE - Non-Critical
        return {
            'hr_initial': int(np.random.normal(85, 15)), 'rr_initial': int(np.random.normal(18, 3)),
            'sbp_min': int(np.random.normal(115, 15)), 'o2sat_min': int(np.random.normal(97, 1))
        }

vitals_data = [generate_vitals(a) for a in acuity]
hr_initial = [v['hr_initial'] for v in vitals_data]
rr_initial = [v['rr_initial'] for v in vitals_data]
sbp_min = [v['sbp_min'] for v in vitals_data]
o2sat_min = [v['o2sat_min'] for v in vitals_data]
temp_initial = np.round(np.random.normal(37.0, 0.5, N_VISITS), 1)

In [ ]:
# --- 4. Unstructured Text Data Generation (Chief Complaint) ---

chief_complaints_map = {
    1: ["Severe crushing chest pain, cannot speak.", "Unresponsive, GCS 3.", "Major trauma, open fracture, severe bleeding."],
    2: ["Shortness of breath for 4 hours, history of asthma.", "New onset of severe headache, worst ever.", "Chest pain radiating to arm, stable vitals."],
    3: ["Abdominal pain x 2 days, vomiting, stable.", "Fever and cough, feels weak, able to ambulate.", "Fall from standing height, minor laceration."],
    4: ["Sore throat, wants rapid test.", "Rash on leg, no fever.", "Mild back pain, needs a prescription refill."],
    5: ["Wants records transferred.", "Dressing change for old wound.", "Routine check-up, non-urgent consultation."]
}

chief_complaint = [random.choice(chief_complaints_map[a]) for a in acuity]
pain = np.random.choice(['10/10 Severe', '8/10 Moderate', '4/10 Mild', '0/10 None'], size=N_VISITS, p=[0.1, 0.3, 0.4, 0.2])

# ICD Code Simulation
def generate_icd(acuity_level):
    if acuity_level <= 2: return random.choice(['I21.0', 'J96.0', 'R57.0']) # STEMI, Resp Failure, Shock
    elif acuity_level == 3: return random.choice(['K56.6', 'J18.9', 'S01.9']) # Obstruction, Pneumonia, Head injury
    else: return random.choice(['J02.9', 'R51', 'M54.5']) # Pharyngitis, Headache, Back pain

icd_code = [generate_icd(a) for a in acuity]

In [ ]:
# --- 5. Create DataFrame ---

data = pd.DataFrame({
    # Identifiers & Static/Tabular
    'subject_id': subject_ids, 'stay_id': stay_ids, 'intime': intime, 'outtime': outtime,
    'gender': gender, 'race': race, 'age': age,
    # Textual Data
    'chief_complaint': chief_complaint, 'pain': pain,
    # Triage/Outcome Targets
    'acuity_mts_level': acuity, 'icu_stay_flag': icu_stay_flag, 'time_to_icu_hrs': time_to_icu_hrs,
    'icd_code': icd_code,
    # Time-Series Proxy Vitals (Initial & Min/Max)
    'hr_initial': hr_initial, 'rr_initial': rr_initial, 'sbp_min': sbp_min,
    'o2sat_min': o2sat_min, 'temp_initial': temp_initial
})

# Display a few rows and the class distribution
print(f"Generated {N_VISITS} Synthetic ED Visits.")
print("\n--- Triage (MTS) Class Distribution ---")
print(data['acuity_mts_level'].value_counts(normalize=True).sort_index() * 100)
print("\n--- Sample Data (First 5 Rows) ---")
print(data.head().to_markdown(index=False))

Generated 10000 Synthetic ED Visits.

--- Triage (MTS) Class Distribution ---
acuity_mts_level
1     0.53
2     4.97
3    34.86
4    45.24
5    14.40
Name: proportion, dtype: float64

--- Sample Data (First 5 Rows) ---
|   subject_id |   stay_id | intime              | outtime                    | gender   | race                   |   age | chief_complaint                                | pain          |   acuity_mts_level |   icu_stay_flag |   time_to_icu_hrs | icd_code   |   hr_initial |   rr_initial |   sbp_min |   o2sat_min |   temp_initial |
|-------------:|----------:|:--------------------|:---------------------------|:---------|:-----------------------|------:|:-----------------------------------------------|:--------------|-------------------:|----------------:|------------------:|:-----------|-------------:|-------------:|----------:|------------:|---------------:|
|        10000 |     20000 | 2150-01-16 22:38:00 | 2150-01-17 03:33:18.032897 | M        | White                 